In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install git+https://github.com/fra31/auto-attack.git

import os
import json
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms as T
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer
from autoattack import AutoAttack

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VisionEncoderDecoderModel.from_pretrained("nlpconnect/vit-gpt2-image-captioning").to(device)
vit_encoder = model.encoder.to(device)
tokenizer = AutoTokenizer.from_pretrained("nlpconnect/vit-gpt2-image-captioning")
feature_extractor = ViTImageProcessor.from_pretrained("nlpconnect/vit-gpt2-image-captioning")

# Also make sure fgsm_attack, pgd_attack, bim_attack, deepfool_attack, wrapped_model are defined.

# Define dataset folder
dataset_folder = "/content/drive/MyDrive/miniproject/subset"

# Paths to images and json file
image_folder = os.path.join(dataset_folder, "Flicker8k_1kSubset")
json_file_path = os.path.join(dataset_folder, "subset_1k_data.json")

# Load the dataset JSON
with open(json_file_path, "r") as f:
    dataset = json.load(f)

  Cloning https://github.com/fra31/auto-attack.git to /tmp/pip-req-build-us8xv7pu
  Running command git clone --filter=blob:none --quiet https://github.com/fra31/auto-attack.git /tmp/pip-req-build-us8xv7pu
  Resolved https://github.com/fra31/auto-attack.git to commit a39220048b3c9f2cca9a4d3a54604793c68eca7e
  Preparing metadata (setup.py) ... done
  Created wheel for autoattack: filename=autoattack-0.1-py3-none-any.whl size=36228 sha256=cb6036cdbc5697b3cdf797ee2c9b23d3627e59fb7e7216168ad255072b91fade
  Stored in directory: /tmp/pip-ephem-wheel-cache-ckbex7t5/wheels/d3/da/df/403e2ecb13ead4fe5da562006891405180c19c91db70e16d55
Successfully built autoattack


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/4.61k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/982M [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

In [3]:


# Output dictionary
results = {}

# FGSM attack
def fgsm_attack(image, epsilon, data_grad):
    perturbed_image = image + epsilon * data_grad.sign()
    return torch.clamp(perturbed_image, 0, 1)

# PGD attack
def pgd_attack(model, images, labels, eps, alpha, iters):
    ori_images = images.clone().detach()
    images = images.clone().detach().requires_grad_(True).to(device)
    for _ in range(iters):
        outputs = model(images, labels=labels)
        loss = outputs.loss
        loss.backward()
        adv_images = images + alpha * images.grad.sign()
        eta = torch.clamp(adv_images - ori_images, min=-eps, max=eps)
        images = torch.clamp(ori_images + eta, min=0, max=1).detach().requires_grad_(True)
        model.zero_grad()
    return images

# DeepFool (approx.)
def deepfool_attack(image, model, num_classes=10, overshoot=0.02, max_iter=50):
    image = image.clone().detach().requires_grad_(True).to(device)
    pert_image = image.clone()
    r_tot = torch.zeros_like(image).to(device)
    x = pert_image.clone().detach().requires_grad_(True)
    loop_i = 0
    while loop_i < max_iter:
        outputs = model(x).last_hidden_state.mean(dim=1)
        label = outputs.argmax(dim=1).item()
        output = outputs[0]
        output[label].backward(retain_graph=True)
        grad_orig = x.grad.data.clone()
        x.grad.zero_()
        min_pert = float("inf")
        for k in range(num_classes):
            if k == label:
                continue
            output[k].backward(retain_graph=True)
            grad_k = x.grad.data.clone()
            x.grad.zero_()
            w_k = grad_k - grad_orig
            f_k = (output[k] - output[label]).detach()
            pert_k = torch.abs(f_k) / (w_k.norm() + 1e-8)
            if pert_k < min_pert:
                min_pert = pert_k
                ri = pert_k * w_k / (w_k.norm() + 1e-8)
        r_tot += ri
        pert_image = image + (1 + overshoot) * r_tot
        x = pert_image.clone().detach().requires_grad_(True)
        new_outputs = model(x).last_hidden_state.mean(dim=1)
        new_label = new_outputs.argmax(dim=1).item()
        if new_label != label:
            break
        loop_i += 1
    return torch.clamp(pert_image, 0, 1)

# BIM attack
def bim_attack(model, image, epsilon=0.03, alpha=0.005, iters=10):
    ori_image = image.clone().detach()
    image.requires_grad = True
    for _ in range(iters):
        outputs = model(image).last_hidden_state.mean(dim=1)
        pred = outputs.argmax(dim=1)
        loss = F.cross_entropy(outputs, pred)
        loss.backward()
        image = image + alpha * image.grad.data.sign()
        eta = torch.clamp(image - ori_image, min=-epsilon, max=epsilon)
        image = torch.clamp(ori_image + eta, min=0, max=1).detach_().requires_grad_(True)
    return image.detach()

# AutoAttack wrapper
class ViTWrapper(torch.nn.Module):
    def __init__(self, vit):
        super().__init__()
        self.vit = vit
        self.linear = torch.nn.Linear(vit.config.hidden_size, 10)

    def forward(self, x):
        out = self.vit(x).last_hidden_state[:, 0, :]
        return self.linear(out)

wrapped_model = ViTWrapper(vit_encoder).to(device)



In [4]:

# Process images in batches
batch_size = 100  # Number of images to process at once

def process_batch(batch_images, batch_filenames, dataset):
    batch_results = []
    for filename, data in zip(batch_filenames, dataset):
        image_path = os.path.join(image_folder, filename)
        image = Image.open(image_path).convert("RGB")
        pixel_values = feature_extractor(images=image, return_tensors="pt").pixel_values.to(device)

        # original caption
        with torch.no_grad():
            orig_caption_ids = model.generate(pixel_values, max_length=16, num_beams=4)
        orig_caption = tokenizer.decode(orig_caption_ids[0], skip_special_tokens=True).strip()

        # attacks...
        # FGSM
        image_tensor = pixel_values.clone().detach().to(device).requires_grad_()
        labels = tokenizer("a photo", return_tensors="pt").input_ids.to(device)
        outputs = model(pixel_values=image_tensor, labels=labels)
        loss = outputs.loss
        model.zero_grad()
        loss.backward()
        fgsm_adv = fgsm_attack(image_tensor, 0.05, image_tensor.grad.data)
        fgsm_caption_ids = model.generate(fgsm_adv, max_length=16, num_beams=4)
        fgsm_caption = tokenizer.decode(fgsm_caption_ids[0], skip_special_tokens=True).strip()

        # PGD
        pgd_adv = pgd_attack(model, pixel_values.clone(), labels, 0.03, 0.005, 10)
        pgd_caption_ids = model.generate(pgd_adv, max_length=16, num_beams=4)
        pgd_caption = tokenizer.decode(pgd_caption_ids[0], skip_special_tokens=True).strip()

        # DeepFool
        deepfool_adv = deepfool_attack(pixel_values.clone(), vit_encoder)
        deepfool_caption_ids = model.generate(deepfool_adv, max_length=16, num_beams=4)
        deepfool_caption = tokenizer.decode(deepfool_caption_ids[0], skip_special_tokens=True).strip()

        # BIM
        bim_adv = bim_attack(vit_encoder, pixel_values.clone(), epsilon=0.03, alpha=0.005, iters=10)
        bim_caption_ids = model.generate(bim_adv, max_length=16, num_beams=4)
        bim_caption = tokenizer.decode(bim_caption_ids[0], skip_special_tokens=True).strip()

        # AutoAttack
        aa_input = pixel_values.clone().detach()
        aa_labels = torch.zeros((aa_input.size(0),), dtype=torch.long, device=device)  # assume class 0
        autoattack = AutoAttack(wrapped_model, norm='Linf', eps=0.1, version='standard', device=device)
        aa_adv = autoattack.run_standard_evaluation(aa_input, aa_labels, bs=batch_size)
        aa_caption_ids = model.generate(aa_adv, max_length=16, num_beams=4)
        aa_caption = tokenizer.decode(aa_caption_ids[0], skip_special_tokens=True).strip()


        # Store results for this image
        image_result = {
            "image": filename,
            "ground_truth_captions": data["captions"],
            "original_caption": orig_caption,
            "fgsm": fgsm_caption,
            "pgd": pgd_caption,
            "BIM": bim_caption,
            "deepfool": deepfool_caption,
            "autoattack": aa_caption
        }
        batch_results.append(image_result)

    return batch_results


# Iterate over dataset in batches
batch_images = []
batch_filenames = []
current_batch = 0

for i, data in enumerate(dataset):
    batch_images.append(data["image"])
    batch_filenames.append(data["image"])

    # Process and save batch
    if len(batch_images) == batch_size or i == len(dataset) - 1:
        batch_results = process_batch(batch_images, batch_filenames, dataset[current_batch:current_batch+batch_size])
        results.update({result["image"]: result for result in batch_results})

        # Save intermediate results
        output_batch_file = f"/content/drive/MyDrive/miniproject/subset/caption_results_batch_{current_batch}.json"
        with open(output_batch_file, "w") as f:
            json.dump(batch_results, f, indent=4)
        print(f"✅ Batch {current_batch} saved to {output_batch_file}")

        # Reset batch data
        batch_images = []
        batch_filenames = []
        current_batch += batch_size

# Save all results to final output file
final_output_file = "/content/drive/MyDrive/miniproject/subset/caption_results.json"
with open(final_output_file, "w") as f:
    json.dump(results, f, indent=4)

print(f"✅ Final JSON saved to {final_output_file}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
You may ignore this warning if your `pad_token_id` (50256) is identical to the `bos_token_id` (50256), `eos_token_id` (50256), or the `sep_token_id` (None), and your input is not padded.
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.

PYDEV DEBUGGER WARNING:
sys.settrace() should not be used when the debugger is being used.
This may cause the debugger to stop working correctly.
If this is needed, please check: 
http://pydev.blogspot.com/2007/06/why-cant-pydev-debugger-work-with.html
to see how to restore th

setting parameters for standard version
using standard version including apgd-ce, apgd-t, fab-t, square.
initial accuracy: 0.00%
max Linf perturbation: 0.00000, nan in tensor: 0, max: 1.00000, min: -0.92941
robust accuracy: 0.00%
setting parameters for standard version
using standard version including apgd-ce, apgd-t, fab-t, square.
initial accuracy: 0.00%
max Linf perturbation: 0.00000, nan in tensor: 0, max: 0.41176, min: -1.00000
robust accuracy: 0.00%
setting parameters for standard version
using standard version including apgd-ce, apgd-t, fab-t, square.
initial accuracy: 0.00%
max Linf perturbation: 0.00000, nan in tensor: 0, max: 0.98431, min: -1.00000
robust accuracy: 0.00%
setting parameters for standard version
using standard version including apgd-ce, apgd-t, fab-t, square.
initial accuracy: 0.00%
max Linf perturbation: 0.00000, nan in tensor: 0, max: 1.00000, min: -0.96078
robust accuracy: 0.00%
setting parameters for standard version
using standard version including apgd-ce

KeyboardInterrupt: 